# Excel explorer

Select a source Excel file from `data/` to inspect its sheets and preview the first sheet.

In [29]:
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import display
import json

pd.set_option("display.max_colwidth", None)  # Don't truncate cell contents
pd.set_option("display.max_columns", None)   # Show all columns
pd.set_option("display.width", None)         # Don't impose a text display width

def find_project_root(start: Path) -> Path:
    """Locate the repository root so this works from Jupyter's chosen cwd."""
    for directory in (start, *start.parents):
        if (directory / 'data').is_dir():
            return directory
    raise FileNotFoundError("Could not find the repository's data directory.")


project_root = find_project_root(Path.cwd().resolve())
excel_files = sorted((project_root / 'data').glob('*.xlsx'))

if not excel_files:
    raise FileNotFoundError(f"No .xlsx files found in {project_root / 'data'}")

file_picker = widgets.Dropdown(
    options=[(file.name, file) for file in excel_files],
    description='Excel file:',
    layout=widgets.Layout(width='700px'),
)
sheet_picker = widgets.Dropdown(description='Sheet:', layout=widgets.Layout(width='400px'))
output = widgets.Output()


In [30]:
def show_preview(*_):
    """Load the selected sheet and display a small, safe preview."""
    selected_file = file_picker.value
    with output:
        output.clear_output(wait=True)
        try:
            frame = pd.read_excel(selected_file, sheet_name=sheet_picker.value, nrows=10)
            print(f"File: {selected_file.name}")
            print(f"Sheet: {sheet_picker.value} | Showing first {len(frame)} rows")
            display(frame)
        except Exception as error:
            print(f"Could not read the selected sheet: {error}")


def update_sheets(*_):
    """Update the sheet dropdown after choosing another workbook."""
    try:
        workbook = pd.ExcelFile(file_picker.value)
        sheet_picker.options = workbook.sheet_names
        sheet_picker.value = workbook.sheet_names[0]
    except Exception as error:
        with output:
            output.clear_output(wait=True)
            print(f"Could not open the selected workbook: {error}")


file_picker.observe(update_sheets, names='value')
sheet_picker.observe(show_preview, names='value')
update_sheets()
display(widgets.VBox([file_picker, sheet_picker, output]))


In [24]:
# Read original file and save schema
original_file = file_picker.value
print(f'Reading from: {original_file}')

generated_dir = project_root / 'data' / 'generated'
generated_dir.mkdir(parents=True, exist_ok=True)

excel_file = pd.ExcelFile(original_file)

original_schema = {}

for sheet in excel_file.sheet_names:
    df = excel_file.parse(sheet)

    columns_info = {
        str(col): str(df[col].dtype)
        for col in df.columns
    }

    original_schema[sheet] = {
        'columns': columns_info,
        'shape': list(df.shape)
    }

schema_file = generated_dir / 'original_schema.json'

with open(schema_file, 'w') as f:
    json.dump(original_schema, f, indent=2)

print(f'Original schema saved to: {schema_file}')

print(f"Schema generated is:")
print(json.dumps(original_schema, indent=2))

Reading from: G:\coding_projects\rekenkamer\data\Algemene Rekenkamer - Casus Data Engineer v1.xlsx
Original schema saved to: G:\coding_projects\rekenkamer\data\generated\original_schema.json
Schema generated is:
{
  "Infoblad": {
    "columns": {
      "Soort meest waarschijnlijke fout en zekerheid": "object",
      "Toelichting": "object",
      "Prioriteit": "float64"
    },
    "shape": [
      8,
      3
    ]
  },
  "RM_uitgaven": {
    "columns": {
      "hoofdstuk nr": "object",
      "hoofdstuk": "object",
      "artikel nr": "float64",
      "artikel": "object",
      "soort artikel": "object",
      "verantwoord bedrag": "float64",
      "steekproef": "object",
      "meest waarschijnlijke fout": "float64",
      "meest waarschijnlijke onzekerheid": "float64",
      "meest waarschijnlijke fout en onzekerheid": "int64",
      "basis onnauwkeurigheid": "float64",
      "nauwkeurigheids verval": "object",
      "maximale fout en onzekerheid": "int64",
      "niet kwantificeerbar

In [26]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

original_file = file_picker.value
generated_dir = project_root / "data" / "generated"
generated_dir.mkdir(parents=True, exist_ok=True)

print(f"Reading from: {original_file}")
print(f"Writing generated files to: {generated_dir}")


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def load_workbook(path: Path) -> dict[str, pd.DataFrame]:
    """Load all sheets from an Excel workbook."""
    return pd.read_excel(path, sheet_name=None)


def copy_workbook(
    workbook: dict[str, pd.DataFrame]
) -> dict[str, pd.DataFrame]:
    """Create an independent copy of every DataFrame in the workbook."""
    return {
        sheet_name: df.copy(deep=True)
        for sheet_name, df in workbook.items()
    }


def save_workbook(
    workbook: dict[str, pd.DataFrame],
    output_file: Path
) -> None:
    """Save all sheets to a new Excel workbook."""
    with pd.ExcelWriter(output_file) as writer:
        for sheet_name, df in workbook.items():
            df.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False
            )


def find_first_numeric_value(
    workbook: dict[str, pd.DataFrame]
):
    """
    Find the first populated numeric cell.

    Returns:
        (sheet_name, row_index, column_name, value)
    """
    for sheet_name, df in workbook.items():
        for column in df.columns:
            if not pd.api.types.is_numeric_dtype(df[column]):
                continue

            valid_values = df[column].dropna()

            if not valid_values.empty:
                row_index = valid_values.index[0]
                value = df.loc[row_index, column]

                return sheet_name, row_index, column, value

    raise ValueError("No populated numeric cell found in workbook.")


def find_first_positive_numeric_value(
    workbook: dict[str, pd.DataFrame]
):
    """
    Find the first populated numeric cell with a value greater than zero.

    Returns:
        (sheet_name, row_index, column_name, value)
    """
    for sheet_name, df in workbook.items():
        for column in df.columns:
            if not pd.api.types.is_numeric_dtype(df[column]):
                continue

            positive_values = df.loc[df[column] > 0, column]

            if not positive_values.empty:
                row_index = positive_values.index[0]
                value = df.loc[row_index, column]

                return sheet_name, row_index, column, value

    raise ValueError("No positive numeric cell found in workbook.")


# ---------------------------------------------------------------------------
# Load original workbook once
# ---------------------------------------------------------------------------

original_workbook = load_workbook(original_file)


# ---------------------------------------------------------------------------
# Store original schema
# ---------------------------------------------------------------------------

original_schema = {}

for sheet_name, df in original_workbook.items():
    original_schema[sheet_name] = {
        "columns": {
            str(column): str(df[column].dtype)
            for column in df.columns
        },
        "shape": list(df.shape)
    }

schema_file = generated_dir / "original_schema.json"

with open(schema_file, "w", encoding="utf-8") as f:
    json.dump(original_schema, f, indent=2)

print(f"Original schema saved to: {schema_file}")


# ---------------------------------------------------------------------------
# diff_schema.xlsx
# ---------------------------------------------------------------------------

def create_diff_schema():
    workbook = copy_workbook(original_workbook)

    sheet_name, _, column, _ = find_first_numeric_value(workbook)
    df = workbook[sheet_name]

    new_name = f"{column}_changed"

    # Ensure the generated name cannot collide with an existing column.
    suffix = 1
    while new_name in df.columns:
        new_name = f"{column}_changed_{suffix}"
        suffix += 1

    df.rename(
        columns={column: new_name},
        inplace=True
    )

    output_file = generated_dir / "diff_schema.xlsx"
    save_workbook(workbook, output_file)

    print(
        f"Created {output_file.name}: "
        f"sheet '{sheet_name}', "
        f"column '{column}' -> '{new_name}'"
    )


# ---------------------------------------------------------------------------
# diff_datatype.xlsx
# ---------------------------------------------------------------------------

def create_diff_datatype():
    workbook = copy_workbook(original_workbook)

    sheet_name, row_index, column, old_value = (
        find_first_numeric_value(workbook)
    )

    new_value = "onbekend"

    df = workbook[sheet_name]
    # Explicitly allow mixed datatypes in this column.
    df[column] = df[column].astype("object")
    df.loc[row_index, column] = new_value

    output_file = generated_dir / "diff_datatype.xlsx"
    save_workbook(workbook, output_file)

    print(
        f"Created {output_file.name}: "
        f"sheet '{sheet_name}', "
        f"row {row_index}, "
        f"column '{column}', "
        f"{old_value!r} -> {new_value!r}"
    )


# ---------------------------------------------------------------------------
# diff_missing.xlsx
# ---------------------------------------------------------------------------

def create_diff_missing():
    workbook = copy_workbook(original_workbook)

    sheet_name, row_index, column, old_value = (
        find_first_numeric_value(workbook)
    )

    new_value = None

    workbook[sheet_name].loc[row_index, column] = new_value

    output_file = generated_dir / "diff_missing.xlsx"
    save_workbook(workbook, output_file)

    print(
        f"Created {output_file.name}: "
        f"sheet '{sheet_name}', "
        f"row {row_index}, "
        f"column '{column}', "
        f"{old_value!r} -> missing"
    )


# ---------------------------------------------------------------------------
# diff_negative.xlsx
# ---------------------------------------------------------------------------

def create_diff_negative():
    workbook = copy_workbook(original_workbook)

    sheet_name, row_index, column, old_value = (
        find_first_positive_numeric_value(workbook)
    )

    new_value = -old_value

    workbook[sheet_name].loc[row_index, column] = new_value

    output_file = generated_dir / "diff_negative.xlsx"
    save_workbook(workbook, output_file)

    print(
        f"Created {output_file.name}: "
        f"sheet '{sheet_name}', "
        f"row {row_index}, "
        f"column '{column}', "
        f"{old_value!r} -> {new_value!r}"
    )


# ---------------------------------------------------------------------------
# Generate variations
# ---------------------------------------------------------------------------

create_diff_schema()
create_diff_datatype()
create_diff_missing()
create_diff_negative()

Reading from: G:\coding_projects\rekenkamer\data\Algemene Rekenkamer - Casus Data Engineer v1.xlsx
Writing generated files to: G:\coding_projects\rekenkamer\data\generated
Original schema saved to: G:\coding_projects\rekenkamer\data\generated\original_schema.json
Created diff_schema.xlsx: sheet 'Infoblad', column 'Prioriteit' -> 'Prioriteit_changed'
Created diff_datatype.xlsx: sheet 'Infoblad', row 0, column 'Prioriteit', np.float64(1.0) -> 'onbekend'
Created diff_missing.xlsx: sheet 'Infoblad', row 0, column 'Prioriteit', np.float64(1.0) -> missing
Created diff_negative.xlsx: sheet 'Infoblad', row 0, column 'Prioriteit', np.float64(1.0) -> np.float64(-1.0)


### This is an example of a basic validation. 
The validation could be triggered by an endpoint that serves as a file upload and immediately gives feedback on the structure of the data to the person who is uploading the file.

In [28]:
from validation import validate_excel_files

generated_dir = project_root / "data" / "generated"

schema_file = generated_dir / "original_schema.json"

files_to_validate = [
    file_picker.value,
    generated_dir / "diff_schema.xlsx",
    generated_dir / "diff_datatype.xlsx",
    generated_dir / "diff_missing.xlsx",
    generated_dir / "diff_negative.xlsx",
]

output_csv = generated_dir / "validation_results.csv"

results = validate_excel_files(
    files=files_to_validate,
    schema_file=schema_file,
    output_csv=output_csv
)

# pd.set_option("display.max_colwidth", None)
display(results)

,file,status,issues
0,Algemene Rekenkamer - Casus Data Engineer v1.xlsx,OK,
1,diff_schema.xlsx,ERROR,Infoblad: missing columns ['Prioriteit'] | Infoblad: unexpected columns ['Prioriteit_changed']
2,diff_datatype.xlsx,ERROR,"Infoblad.Prioriteit: dtype expected float64, found object"
3,diff_missing.xlsx,OK,
4,diff_negative.xlsx,OK,
